## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

In [46]:
# (1) Preferred in notebooks:
%pip install nest_asyncio


Note: you may need to restart the kernel to use updated packages.


c:\Users\saket\projects\agents\.venv\Scripts\python.exe: No module named pip


In [50]:
import nest_asyncio
nest_asyncio.apply()


In [51]:
# The imports

from dotenv import load_dotenv
from agents import Agent, Runner, trace



In [52]:
# The usual starting point

load_dotenv(override=True)


True

In [53]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-4o-mini")

In [54]:
agent

Agent(name='Jokester', instructions='You are a joke teller', handoff_description=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

In [5]:
help(agent)

Help on Agent in module agents.agent object:

class Agent(typing.Generic)
 |  Agent(name: 'str', instructions: 'str | Callable[[RunContextWrapper[TContext], Agent[TContext]], MaybeAwaitable[str]] | None' = None, handoff_description: 'str | None' = None, handoffs: 'list[Agent[Any] | Handoff[TContext]]' = <factory>, model: 'str | Model | None' = None, model_settings: 'ModelSettings' = <factory>, tools: 'list[Tool]' = <factory>, mcp_servers: 'list[MCPServer]' = <factory>, mcp_config: 'MCPConfig' = <factory>, input_guardrails: 'list[InputGuardrail[TContext]]' = <factory>, output_guardrails: 'list[OutputGuardrail[TContext]]' = <factory>, output_type: 'type[Any] | AgentOutputSchemaBase | None' = None, hooks: 'AgentHooks[TContext] | None' = None, tool_use_behavior: "Literal['run_llm_again', 'stop_on_first_tool'] | StopAtTools | ToolsToFinalOutputFunction" = 'run_llm_again', reset_tool_choice: 'bool' = True) -> None
 |
 |  An agent is an AI model configured with instructions, tools, guardrails

In [57]:
from agents import Agent, Runner

agent = Agent(
    name="StreamingJokester",
    instructions="Tell a funny joke, one line at a time."
)

async def stream_agent():
    # no await here, run_streamed is sync
    stream = Runner.run_streamed(agent, "Tell a joke about AI agents")
    async for event in stream.iter_events():
        print("🔹 Event:", event)


In [60]:
# 1️⃣ Define your agent and streamer
from agents import Agent, Runner

agent = Agent(
    name="StreamingJokester",
    instructions=" Tell a funny joke, one line at a time."
)

async def stream_agent():
    # run_streamed is a sync call returning RunResultStreaming
    stream = Runner.run_streamed(agent, "Tell a joke about AI agents")
    # use .stream_events(), which yields an async iterator of events
    async for event in stream.stream_events():
        print("🔹 Event:", event)


In [61]:
# 2️⃣ (In a fresh cell) — if you run into "already running event loop" errors,
# apply nest_asyncio before you fire off the streamer:
import nest_asyncio
nest_asyncio.apply()

await stream_agent()


🔹 Event: AgentUpdatedStreamEvent(new_agent=Agent(name='StreamingJokester', instructions='Tell a funny joke, one line at a time.', handoff_description=None, handoffs=[], model=None, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), type='agent_updated_stream_event')
🔹 Event: RawResponsesStreamEvent(data=ResponseCreatedEvent(response=Response(id='resp_6892d99fd9ac819296e9a6bc82202534051683ea516ba54e', created_at=1754454431.0, error=None, incomplete_details=None, instructions='Tell a funny joke, one line at a time.', metadata={}, model='gpt-4o-2024-08-06', object='respo

In [68]:
# 1️⃣ Patch the running event loop (only if you hit "already running loop" errors)
import nest_asyncio
nest_asyncio.apply()

# 2️⃣ Import & define your agent
from agents import Agent, Runner

agent = Agent(
    name="StreamingJokester",
    instructions="Tell a funny joke, one line at a time."
)

# 3️⃣ Define a streamer that prints *only* the text deltas
async def stream_text_only():
    stream = Runner.run_streamed(agent, "Tell a joke about AI agents")
    async for ev in stream.stream_events():
        # some events wrap their payload in `.data`, others expose `.delta` directly
        payload = getattr(ev, "data", ev)
        delta   = getattr(payload, "delta", None)
        if delta is not None:
            print(delta, end="", flush=True)
    print()  # final newline

# 4️⃣ Fire it off in your notebook
await stream_text_only()


Why did the AI agent break up with the robot?


In [67]:
# 1️⃣ (Optional) patch the running loop if you hit "already running" errors
import nest_asyncio
nest_asyncio.apply()

import asyncio
from agents import Agent, Runner

agent = Agent(
    name="StreamingJokester",
    instructions="Tell a funny joke, one line at a time."
)

async def stream_text_only(delay: float = 0.1):
    """
    Streams just the text deltas from the agent,
    pausing `delay` seconds between each chunk so you can watch.
    """
    stream = Runner.run_streamed(agent, "Tell a joke about AI agents")
    async for ev in stream.stream_events():
        payload = getattr(ev, "data", ev)
        delta   = getattr(payload, "delta", None)
        if delta is not None:
            print(delta, end="", flush=True)
            await asyncio.sleep(delay)
    print()  # newline at end

# Kick it off with a 0.2s pause between deltas:
await stream_text_only(delay=0.2)


Why did the AI agent break up with its algorithm?


In [49]:
# in a notebook cell
!uv pip install nest_asyncio


Using Python 3.12.11 environment at: C:\Users\saket\projects\agents\.venv
Audited 1 package in 662ms


In [7]:
# Run the joke with Runner.run(agent, prompt) then print final_output

with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)

Why did the autonomous AI agent bring a ladder to work?

Because it wanted to reach new heights in automation!


In [31]:
# Run the joke with Runner.run(agent, prompt) then print final_output

with trace("Telling a joke"):
    result =  Runner.run_sync(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)

RuntimeError: This event loop is already running

In [12]:
result

RunResult(input='Tell a joke about Autonomous AI Agents', new_items=[MessageOutputItem(agent=Agent(name='Jokester', instructions='You are a joke teller', handoff_description=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), raw_item=ResponseOutputMessage(id='msg_6892d3469f60819fb2620f5273d4005d0c343e531ea427a9', content=[ResponseOutputText(annotations=[], text='Why did the autonomous AI agent bring a ladder to work?\n\nBecause it wanted to reach new heights in automation!', type='output_text', logprobs=[])], role='assistant', s

In [13]:
print(type(result))


<class 'agents.result.RunResult'>


In [14]:
import pprint
pprint.pprint(result.__dict__)


{'_last_agent': Agent(name='Jokester',
                      instructions='You are a joke teller',
                      handoff_description=None,
                      handoffs=[],
                      model='gpt-4o-mini',
                      model_settings=ModelSettings(temperature=None,
                                                   top_p=None,
                                                   frequency_penalty=None,
                                                   presence_penalty=None,
                                                   tool_choice=None,
                                                   parallel_tool_calls=None,
                                                   truncation=None,
                                                   max_tokens=None,
                                                   reasoning=None,
                                                   metadata=None,
                                                   store=None,
                

In [15]:
for key, value in vars(result).items():
    print(f"{key}:\n  {value}\n")



input:
  Tell a joke about Autonomous AI Agents

new_items:
  [MessageOutputItem(agent=Agent(name='Jokester', instructions='You are a joke teller', handoff_description=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), raw_item=ResponseOutputMessage(id='msg_6892d3469f60819fb2620f5273d4005d0c343e531ea427a9', content=[ResponseOutputText(annotations=[], text='Why did the autonomous AI agent bring a ladder to work?\n\nBecause it wanted to reach new heights in automation!', type='output_text', logprobs=[])], role='assistant', status=

In [16]:
result

RunResult(input='Tell a joke about Autonomous AI Agents', new_items=[MessageOutputItem(agent=Agent(name='Jokester', instructions='You are a joke teller', handoff_description=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), raw_item=ResponseOutputMessage(id='msg_6892d3469f60819fb2620f5273d4005d0c343e531ea427a9', content=[ResponseOutputText(annotations=[], text='Why did the autonomous AI agent bring a ladder to work?\n\nBecause it wanted to reach new heights in automation!', type='output_text', logprobs=[])], role='assistant', s

In [17]:
print("Final Output:", result.final_output)
print("Input:", result.input)
print("Raw Responses:", result.raw_responses)
print("Agent Name:", result._last_agent.name)


Final Output: Why did the autonomous AI agent bring a ladder to work?

Because it wanted to reach new heights in automation!
Input: Tell a joke about Autonomous AI Agents
Raw Responses: [ModelResponse(output=[ResponseOutputMessage(id='msg_6892d3469f60819fb2620f5273d4005d0c343e531ea427a9', content=[ResponseOutputText(annotations=[], text='Why did the autonomous AI agent bring a ladder to work?\n\nBecause it wanted to reach new heights in automation!', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], usage=Usage(requests=1, input_tokens=23, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=23, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=46), response_id='resp_6892d346547c819fae426702dce99ca30c343e531ea427a9')]
Agent Name: Jokester


In [21]:
print(result.raw_responses[0].output)

[ResponseOutputMessage(id='msg_6892d3469f60819fb2620f5273d4005d0c343e531ea427a9', content=[ResponseOutputText(annotations=[], text='Why did the autonomous AI agent bring a ladder to work?\n\nBecause it wanted to reach new heights in automation!', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')]


In [22]:
with trace("Telling a joke") as tracer:
    result = await Runner.run(agent, "Tell a joke…")
    print("Joke:", result.final_output)

# Now introspect
print(sorted(attr for attr in dir(tracer) if not attr.startswith("_")))
#test
#test1


Joke: Why did the scarecrow win an award? 

Because he was outstanding in his field!
['export', 'finish', 'group_id', 'metadata', 'name', 'start', 'trace_id']


In [23]:
# List of attributes you discovered
attrs = ["export", "finish", "group_id", "metadata", "name", "start", "trace_id"]

# Iterate and print each one
for attr in attrs:
    value = getattr(tracer, attr, None)
    # If it’s a method, call it (where sensible), otherwise just show the value
    if callable(value) and attr not in ("finish",):  # avoid re‐finishing the trace
        try:
            out = value()   # e.g. export() → dict/json
        except TypeError:
            out = value     # method needs args, so just show the bound method
        print(f"{attr}(): {out!r}")
    else:
        print(f"{attr}: {value!r}")


export(): {'object': 'trace', 'id': 'trace_729a543e2c264465b783e24ae6bc4c71', 'workflow_name': 'Telling a joke', 'group_id': None, 'metadata': None}
finish: <bound method TraceImpl.finish of <agents.tracing.traces.TraceImpl object at 0x0000016E215902F0>>
group_id: None
metadata: None
name: 'Telling a joke'
start(): None
trace_id: 'trace_729a543e2c264465b783e24ae6bc4c71'


In [24]:
full = tracer.get_trace()


AttributeError: 'TraceImpl' object has no attribute 'get_trace'

{'__name__': '__main__', '__doc__': 'Automatically created module for IPython interactive environment', '__package__': None, '__loader__': None, '__spec__': None, '__builtin__': <module 'builtins' (built-in)>, '__builtins__': <module 'builtins' (built-in)>, '_ih': ['', '# At the top level of your script or REPL:\nx = 42\ndef greet(name):\n    return f"Hello, {name}!"\n\nprint(globals())'], '_oh': {}, '_dh': [WindowsPath('c:/Users/saket/projects/agents/1_foundations')], 'In': ['', '# At the top level of your script or REPL:\nx = 42\ndef greet(name):\n    return f"Hello, {name}!"\n\nprint(globals())'], 'Out': {}, 'get_ipython': <bound method InteractiveShell.get_ipython of <ipykernel.zmqshell.ZMQInteractiveShell object at 0x000001A1B3034B00>>, 'exit': <IPython.core.autocall.ZMQExitAutocall object at 0x000001A1B2D0DBE0>, 'quit': <IPython.core.autocall.ZMQExitAutocall object at 0x000001A1B2D0DBE0>, 'open': <function open at 0x000001A1B112F9C0>, '_': '', '__': '', '___': '', '__vsc_ipynb_fi

dict

'Hello, Alice!'

## Now go and look at the trace

https://platform.openai.com/traces